In [2]:
#my kernel doesnt want to sustain over multiple sessions, working on it
!pip install datasets

  Using cached datasets-5.0.0-py3-none-any.whl.metadata (23 kB)
  Using cached filelock-3.29.4-py3-none-any.whl.metadata (2.0 kB)
  Using cached numpy-2.4.6-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached pyarrow-24.0.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached pandas-3.0.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached tqdm-4.68.2-py3-none-any.whl.metadata (58 kB)
  Using cached xxhash-3.7.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (13 kB)
  Using cached multiprocess-0.70.19-py312-none-any.whl.metadata (7.5 kB)
  Using cached fsspec-2026.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached huggingface_hub-1.19.0-py3-none-any.whl.metad

Inspecting DaNe_plus dataset

In [1]:
from collections import Counter
from datasets import load_dataset

ds = load_dataset("KennethEnevoldsen/dane_plus")
print(ds)

ModuleNotFoundError: No module named 'datasets'

In [7]:
for split in ds:
    print(split, len(ds[split]))

for split in ds:
    labels = set()

    for ex in ds[split]:
        for ent in ex["ents"]:
            labels.add(ent["label"])

    print(split, sorted(labels))

train 4383
test 565
dev 564
train ['LOC', 'MISC', 'ORG', 'PER']
test ['CARDINAL', 'DATE', 'EVENT', 'FACILITY', 'GPE', 'LAW', 'LOC', 'LOCATION', 'MISC', 'MONEY', 'NORP', 'ORDINAL', 'ORG', 'ORGANIZATION', 'PER', 'PERCENT', 'PERSON', 'PRODUCT', 'QUANTITY', 'TIME', 'WORK OF ART', 'WORK_OF_ART']
dev ['LOC', 'MISC', 'ORG', 'PER']


So far, only the test split actually contains all of the labels I want to assign

Inspecting if I did not mess it up when loading dane dataset and preprocessing it to corpus - it could be pulling from there

Checking if DaNe plus splits are with containing same labels

In [8]:
for split in ds:
    print(ds[split].info)

DatasetInfo(features={'text': Value('string'), 'ents': List({'end': Value('int64'), 'label': Value('string'), 'start': Value('int64')}), 'sents': List({'end': Value('int64'), 'start': Value('int64')}), 'tokens': List({'dep': Value('string'), 'end': Value('int64'), 'head': Value('int64'), 'id': Value('int64'), 'lemma': Value('string'), 'morph': Value('string'), 'pos': Value('string'), 'start': Value('int64'), 'tag': Value('string')})}, builder_name='parquet', dataset_name='dane_plus', config_name='default', version=0.0.0, splits={'train': SplitInfo(name='train', num_bytes=7928995, num_examples=4383, dataset_name='dane_plus'), 'test': SplitInfo(name='test', num_bytes=996467, num_examples=565, dataset_name='dane_plus'), 'dev': SplitInfo(name='dev', num_bytes=1021780, num_examples=564, dataset_name='dane_plus')}, download_size=1627548, dataset_size=9947242, size_in_bytes=11574790)
DatasetInfo(features={'text': Value('string'), 'ents': List({'end': Value('int64'), 'label': Value('string'), 

In [ ]:
label_counter = Counter()

for split in ["train", "dev", "test"]:
    for ex in ds[split]:
        for ent in ex["ents"]:
            label_counter[ent["label"]] += 1

print(label_counter)

/work/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Counter({'PER': 1421, 'MISC': 1123, 'LOC': 1057, 'ORG': 901, 'PERSON': 175, 'ORGANIZATION': 145, 'GPE': 77, 'DATE': 67, 'CARDINAL': 64, 'NORP': 51, 'EVENT': 18, 'MONEY': 15, 'FACILITY': 13, 'TIME': 11, 'WORK OF ART': 9, 'QUANTITY': 7, 'ORDINAL': 7, 'WORK_OF_ART': 7, 'LOCATION': 6, 'PRODUCT': 4, 'LAW': 3, 'PERCENT': 2})


checking if it is same across splits

In [4]:
from collections import defaultdict

split_labels = defaultdict(set)

for split in ["train", "dev", "test"]:
    for ex in ds[split]:
        for ent in ex["ents"]:
            split_labels[split].add(ent["label"])

for split, labels in split_labels.items():
    print(split, sorted(labels))

train ['LOC', 'MISC', 'ORG', 'PER']
dev ['LOC', 'MISC', 'ORG', 'PER']
test ['CARDINAL', 'DATE', 'EVENT', 'FACILITY', 'GPE', 'LAW', 'LOC', 'LOCATION', 'MISC', 'MONEY', 'NORP', 'ORDINAL', 'ORG', 'ORGANIZATION', 'PER', 'PERCENT', 'PERSON', 'PRODUCT', 'QUANTITY', 'TIME', 'WORK OF ART', 'WORK_OF_ART']


In [3]:
for split in ["train", "dev", "test"]:
    print(f"\n{split}")
    for ex in ds[split]:
        if ex["ents"]:
            print(ex["ents"][:5])
            break


train
[{'end': 17, 'label': 'ORG', 'start': 14}, {'end': 53, 'label': 'LOC', 'start': 44}, {'end': 99, 'label': 'PER', 'start': 82}]

dev
[{'end': 22, 'label': 'PER', 'start': 12}]

test
[{'end': 18, 'label': 'NORP', 'start': 10}, {'end': 49, 'label': 'PERSON', 'start': 31}, {'end': 65, 'label': 'PERSON', 'start': 53}, {'end': 87, 'label': 'GPE', 'start': 80}]


split in my corpus

In [6]:
from pathlib import Path
import spacy
from spacy.tokens import DocBin

nlp = spacy.blank("da")
base = Path("corpus/cdt_ddt")

def load_docs(path):
    return list(DocBin().from_disk(path).get_docs(nlp.vocab))

def get_labels(docs):
    labels = set()
    for doc in docs:
        for ent in doc.ents:
            labels.add(ent.label_)
    return sorted(labels)

for split in ["train","dev","test","data"]:
    path = base / f"{split}.spacy"
    docs = load_docs(path)
    print(split, get_labels(docs))
    print(docs[0])

from collections import Counter
from spacy.tokens import Doc

Doc.set_extension("split", default=None, force=True)
db = DocBin(store_user_data=True).from_disk("corpus/cdt_ddt/data.spacy")
docs = list(db.get_docs(nlp.vocab))

splits = Counter(doc._.split for doc in docs)
print(splits)



train ['LOC', 'MISC', 'ORG', 'PER']
På fredag har SID inviteret til reception i SID-huset i anledning af at formanden Kjeld Christensen går ind i de glade tressere. Dagen er ganske vist først 21. januar, men når mennesker skal mødes sker det mest praktisk på en fredag, hvis andre end de sædvanlige receptionsløver skal møde frem. Er Køvenhavner Det er få der ved at Kjeld Christensen er født i København, bob,bob! Men han blev da reddet for Jylland, da hele hans opvækst skete i Århus. Som 14-årig kom Kjeld Christensen til landbruget, men efter soldatertiden kom han på en fabrik, hvor han arbejdede i 18 år. Da Korsholmskolen i Hinnerup søgte en ny pedel fik Kjeld Christensen pladsen. Der arbejdede han de næste 10 år. I otte år var han med i byrådet i Hinnerup valgt af socialdemokratiet, han er tit blevet opfordret til at lade sig opstille igen, efter han er flyttet til Hadsten, men formanden for SID ved at en afdelingsformand skal repræsentere alle, og selvom socialdemokraterne er i overta

so.. only cdt_ddt/data.spacy has entites to analyze :/ the others is just cdt data

fixed :)